# Rama 3 (punto C): entrenamiento en Kaggle

Fine-tuning de YOLO11m (COCO) sobre las imágenes fisheye de WoodScape, **sin rectificar**, con **30 épocas**, `imgsz=640` y `batch=16`. La rama 4 (cilíndrica + fine-tuning) tiene que usar las mismas 30 épocas para que las dos se puedan comparar. Usa el mismo código que `notebooks/rama3_finetune.ipynb` (`fisheye_aug.py` y `tools/train_rama3.py`): acá solo se entrena; la evaluación se hace en la Mac con el `best.pt`.

**Antes de correr** (panel derecho, *Session options* / *Settings*):
1. **Accelerator:** GPU T4 x2. Con dos GPUs la corrida se reparte en las dos (DDP) y tarda casi la mitad.
2. **Internet:** activado, para instalar Ultralytics y bajar `yolo11m.pt`. Kaggle puede pedir verificar la cuenta con el teléfono.
3. **Add Input:** el dataset `subarnadasgupta/woodscapes` y el dataset privado donde subiste `rama3_kaggle.zip`.

**Cómo correrlo:**
1. Con `PRUEBA = True`, *Run All*. Tarda unos minutos: verifica las imágenes por sha256, entrena 1 época con el 5% del train y estima cuánto va a tardar la corrida completa.
2. Si todo da bien, cambiar a `PRUEBA = False` y *Save Version* → *Save & Run All (Commit)*. Corre solo en segundo plano; se puede cerrar el navegador.
3. Al terminar, en la pestaña *Output* de esa versión, bajar `rama3_base_kaggle.zip` y descomprimirlo en `outputs/rama3/kaggle/` del proyecto.

In [ ]:
!pip install -q ultralytics==8.4.164

import glob
import hashlib
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import torch
import ultralytics

print(f"ultralytics {ultralytics.__version__} | torch {torch.__version__} | python {sys.version.split()[0]}")
print(f"GPUs: {torch.cuda.device_count()} -> {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")

### Configuración

Solo hay que tocar `PRUEBA`. El resto está fijo para que coincida con la rama 4 y con la notebook de la Mac.

In [ ]:
PRUEBA = True  # True: chequeo corto (1 epoca, 5% del train). False: el entrenamiento real de 30 epocas

EPOCHS = 1 if PRUEBA else 30  # el mismo numero de epocas que la rama 4
FRACCION = 0.05 if PRUEBA else 1.0
CLOSE_MOSAIC = 0 if PRUEBA else 10  # en la prueba mantengo el mosaico para medir la velocidad real
BATCH = 16  # batch total: con dos GPUs son 8 por GPU, el mismo batch efectivo que en la Mac
IMGSZ = 640
WORKERS = 4
N_GPU = torch.cuda.device_count()
DEVICE = ",".join(str(i) for i in range(N_GPU)) if N_GPU else "cpu"  # "0,1" con T4 x2: una corrida en las dos
TRABAJO = Path("/kaggle/working")

assert N_GPU > 0, "no hay GPU: elegir Accelerator = GPU T4 x2 en las opciones de la sesion"
print(f"prueba: {PRUEBA} | epocas: {EPOCHS} | fraccion: {FRACCION} | device: {DEVICE} | batch: {BATCH} | imgsz: {IMGSZ}")

### Datos y código

Busco en `/kaggle/input` la carpeta `rgb_images` de WoodScape y el paquete del repo (`fisheye_aug.py`, el script de entrenamiento, `config/`, el manifest y las etiquetas de `convert_to_yolo.py`). Si Kaggle no descomprimió el zip, lo descomprimo acá. El paquete se copia a `/kaggle/working/repo` porque `/kaggle/input` es de solo lectura.

In [ ]:
def buscar_archivo(nombre, raiz="/kaggle/input"):
    for d, _, archivos in os.walk(raiz):
        if nombre in archivos:
            return Path(d) / nombre
    return None


def buscar_imagenes(raiz="/kaggle/input"):
    # la carpeta rgb_images que tiene las png adentro (el dataset puede venir con carpetas anidadas)
    for d, subdirs, _ in os.walk(raiz):
        if "rgb_images" in subdirs and glob.glob(os.path.join(d, "rgb_images", "*.png")):
            return Path(d) / "rgb_images"
    return None


fa = buscar_archivo("fisheye_aug.py")
if fa is None:
    z = buscar_archivo("rama3_kaggle.zip")
    assert z is not None, "no encuentro el paquete: agregar como input el dataset con rama3_kaggle.zip"
    with zipfile.ZipFile(z) as f:
        f.extractall(TRABAJO / "paquete")
    fa = buscar_archivo("fisheye_aug.py", TRABAJO / "paquete")
REPO = TRABAJO / "repo"
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(fa.parent, REPO)

IMAGENES = buscar_imagenes()
assert IMAGENES is not None, "no encuentro rgb_images: agregar como input el dataset subarnadasgupta/woodscapes"
n_png = len(list(IMAGENES.glob("*.png")))
n_lbl = len(list((REPO / "yolo_ds/labels").glob("*.txt")))
print(f"repo: {REPO} | imagenes: {IMAGENES} ({n_png} png) | etiquetas: {n_lbl}")
assert n_png == 8234 and n_lbl == 8233, (n_png, n_lbl)

### Verificación de las imágenes

Regla del equipo: entrenar solo con archivos idénticos a los del manifest. Comparo el sha256 de las 7233 imágenes de train y val de Kaggle contra `manifest.csv`. Lo hago solo en la prueba; en el commit ya está verificado.

In [ ]:
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


if PRUEBA:
    man = pd.read_csv(REPO / "splits/manifest.csv")
    pool = man[man["split"] == "train_pool"]
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex:
        hashes = list(ex.map(sha256, [IMAGENES / f for f in pool["filename"]]))
    distintas = (pd.Series(hashes, index=pool.index) != pool["sha256"]).sum()
    print(f"train_pool: {len(pool)} imagenes | distintas al manifest: {distintas} | {time.time() - t0:.0f} s")
    assert distintas == 0, "hay imagenes distintas al manifest: avisar al equipo antes de entrenar"
else:
    print("verificacion hecha en la prueba")

### Entrenamiento

Llama a `tools/train_rama3.py`, que arma `images/` + `labels/` con enlaces a las imágenes (sin copiarlas), controla los números del equipo (6148 / 1085 / 1000 imágenes y 74.909 / 13.176 / 11.910 cajas) y entrena la variante `base` con el trainer de Ultralytics sin cambios. Ultralytics guarda `best.pt` (mejor mAP@50-95 en val, el criterio del equipo) y `last.pt` en cada época.

In [ ]:
cmd = [
    sys.executable, str(REPO / "tools/train_rama3.py"), "--variante", "base", "--epochs", str(EPOCHS),
    "--device", DEVICE, "--imagenes", str(IMAGENES), "--etiquetas", str(REPO / "yolo_ds/labels"),
    "--listas", str(REPO / "yolo_ds"), "--ds", str(TRABAJO / "ds_base"), "--salida", str(TRABAJO / "runs"),
    "--modelo", "yolo11m.pt", "--imgsz", str(IMGSZ), "--batch", str(BATCH), "--workers", str(WORKERS),
    "--fraccion", str(FRACCION), "--close-mosaic", str(CLOSE_MOSAIC), "--cache", "ram",
]
print(" ".join(cmd), flush=True)

LOG = TRABAJO / "train_base.log"
t0 = time.time()
with open(LOG, "w") as log:
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:
        print(linea, end="", flush=True)
        log.write(linea)
    p.wait()
DURACION = time.time() - t0
print(f"\ncodigo de salida {p.returncode} | {DURACION / 60:.1f} min")
assert p.returncode == 0, f"el entrenamiento fallo, ver {LOG}"

### Resumen

In [ ]:
RUN = TRABAJO / "runs/rama3_base"
df = pd.read_csv(RUN / "results.csv")
df.columns = df.columns.str.strip()
mejor = df.loc[df["metrics/mAP50-95(B)"].idxmax()]
print(f"epocas: {len(df)} | mejor epoca {int(mejor['epoch'])}: mAP50-95 {mejor['metrics/mAP50-95(B)']:.4f} | mAP50 {mejor['metrics/mAP50(B)']:.4f} (val, ultralytics)")
print(f"best.pt: {(RUN / 'weights/best.pt').is_file()} | last.pt: {(RUN / 'weights/last.pt').is_file()}")

if PRUEBA:
    # estimacion de la corrida completa a partir de las barras de progreso de la prueba
    texto = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", LOG.read_text()).replace("\r", "\n")

    def velocidad(m):
        v = float(m.group(1))
        return v if m.group(2) == "it/s" else 1 / v

    tren = [velocidad(m) for m in re.finditer(r"640: .*?\d+/\d+ ([\d.]+)(it/s|s/it)", texto)]
    val = [float(m.group(1)) for m in re.finditer(r"mAP50-95\): 100%.*?\d+/\d+ [\d.]+(?:it/s|s/it) ([\d.]+)s", texto)]
    if tren and val:
        it_epoca = -(-6148 // BATCH)
        seg_epoca = it_epoca / tren[-1] + val[-1]
        print(f"velocidad: {tren[-1]:.2f} it/s | val: {val[-1]:.0f} s -> una epoca completa ~{seg_epoca / 60:.1f} min")
        print(f"30 epocas: ~{30 * seg_epoca / 3600:.1f} h (mas unos minutos de arranque)")
    else:
        print(f"no pude leer la velocidad del log; la prueba tardo {DURACION / 60:.1f} min en total")
else:
    print(f"entrenamiento completo en {DURACION / 3600:.2f} h")
df.tail(5)

### Empaquetado para bajar

Junto en `rama3_base_kaggle.zip` los pesos, el `results.csv`, los gráficos de Ultralytics y el log, con la misma estructura que espera la notebook de la Mac: descomprimido en `outputs/rama3/kaggle/` queda `outputs/rama3/kaggle/rama3_base/weights/best.pt`. Después borro las carpetas de trabajo (enlaces y copias), así la salida de Kaggle queda chica.

In [ ]:
ENTREGA = TRABAJO / "entrega"
if ENTREGA.exists():
    shutil.rmtree(ENTREGA)
shutil.copytree(RUN, ENTREGA / "rama3_base")
shutil.copy(LOG, ENTREGA / "rama3_base" / LOG.name)
zip_final = shutil.make_archive(str(TRABAJO / "rama3_base_kaggle"), "zip", ENTREGA)

for d in (TRABAJO / "ds_base", REPO, TRABAJO / "paquete", ENTREGA):
    shutil.rmtree(d, ignore_errors=True)
print(f"{zip_final}: {Path(zip_final).stat().st_size / 1e6:.0f} MB")
print(sorted(p.name for p in TRABAJO.iterdir()))